# 09 — Output Generation

Assemble and validate the three submission deliverables (`File_1.csv`, `File_2.csv`, `File_3.csv`) in the exact schema required by the datathon brief.

### Compliance checklist (all enforced programmatically below)
| # | Rule | Source |
|---|------|--------|
| 1 | Exact column names per schema | Brief §4 |
| 2 | `total_ev_projected_2027 == 2,498,159` | Mandatory SARIMA output |
| 3 | `estimated_demand_kw = n_chargers × 150` | Fixed charger power |
| 4 | File_3 grid_status ∈ {Moderate, Congested} — **no Sufficient** | Brief §4.3 |
| 5 | File_1 counts consistent with File_2 / File_3 row counts | Internal consistency |
| 6 | File_2 grid_status ∈ {Sufficient, Moderate, Congested} | Brief §4.2 |
| 7 | File_3 distributor_network ∈ {i-DE, Endesa, Viesgo} | Brief §4.3 |
| 8 | Coordinates within Spain mainland bounding box | Sanity check |

## Data Inputs
| File | Source | Description |
|------|--------|-------------|
| `data/processed/stations_with_grid_status.csv` | NB08 | All proposed stations with grid viability classification |
| `data/processed/friction_points.csv` | NB08 | Subset: Moderate + Congested only |
| `data/processed/baseline_kpi.csv` | NB04 | Existing interurban fast-charger count (≥50 kW) |
| `data/processed/grid_consolidated.csv` | NB05 | 2,147 safely consolidated substations (for DSO summary) |
| `data/processed/station_validation_metrics.csv` | NB07b | Utilization & queue risk per station (for summary) |

## Data Outputs
| File | Schema | Description |
|------|--------|-------------|
| `output/File_1.csv` | 1 row, 4 cols | Global Network KPIs |
| `output/File_2.csv` | N rows, 6 cols | All proposed charging locations |
| `output/File_3.csv` | M rows, 7 cols | Friction points (grid-constrained locations) |

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    DATA_DIR = Path('../data/processed')
    OUTPUT_DIR = Path('../output')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR = Path('data/processed')
    OUTPUT_DIR = Path('output')

from src.constants import (
    FILE_1_COLUMNS, FILE_2_COLUMNS, FILE_3_COLUMNS,
    POWER_PER_CHARGER_KW, EV_FLEET_2027,
    VALID_GRID_STATUSES_FILE2, VALID_GRID_STATUSES_FILE3,
    VALID_DISTRIBUTORS,
    GRID_SUFFICIENT_MIN_MW, GRID_MODERATE_MIN_MW,
    MAX_STATION_SPACING_TENT_CORE_KM, MAX_STATION_SPACING_TENT_COMP_KM,
    MAX_STATION_SPACING_KM,
    DEFAULT_STATUS_IF_NO_SUBSTATION,
    MIN_EXISTING_CHARGER_POWER_KW,
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Imports OK')
print(f'   Mandatory total_ev_projected_2027: {EV_FLEET_2027:,}')
print(f'   Power per charger: {POWER_PER_CHARGER_KW} kW')
print()
print(f'   File_1 columns: {FILE_1_COLUMNS}')
print(f'   File_2 columns: {FILE_2_COLUMNS}')
print(f'   File_3 columns: {FILE_3_COLUMNS}')
print()
print(f'   Valid File_2 grid statuses: {VALID_GRID_STATUSES_FILE2}')
print(f'   Valid File_3 grid statuses: {VALID_GRID_STATUSES_FILE3}')
print(f'   Valid distributors: {VALID_DISTRIBUTORS}')

## Step 1: Load inputs

In [ ]:
# ── Primary inputs (NB08) ─────────────────────────────────────────────────────
stations = pd.read_csv(DATA_DIR / 'stations_with_grid_status.csv')
print(f'📍 Proposed stations (NB08): {len(stations):,}')
print(f'   Columns: {list(stations.columns)}')

friction = pd.read_csv(DATA_DIR / 'friction_points.csv')
print(f'🔥 Friction points  (NB08): {len(friction):,}')

# ── Existing stations baseline (NB04) ────────────────────────────────────────
baseline_path = DATA_DIR / 'baseline_kpi.csv'
interurban_path = DATA_DIR / 'interurban_chargers_baseline.csv'
if baseline_path.exists():
    baseline = pd.read_csv(baseline_path)
    if 'total_existing_stations_baseline' in baseline.columns:
        total_existing = int(baseline['total_existing_stations_baseline'].iloc[0])
    elif len(baseline) == 1:
        total_existing = int(baseline.iloc[0, 0])
    else:
        total_existing = int((pd.read_csv(interurban_path)['max_power_kw'] >= MIN_EXISTING_CHARGER_POWER_KW).sum()) if interurban_path.exists() else 3246
elif interurban_path.exists():
    interurban = pd.read_csv(interurban_path)
    total_existing = int((interurban['max_power_kw'] >= MIN_EXISTING_CHARGER_POWER_KW).sum())
else:
    total_existing = 3246  # Known fast-charger baseline from NB04
print(f'📊 Existing stations baseline: {total_existing:,}')

# ── Supplementary data (for executive summary) ───────────────────────────────
grid_path = DATA_DIR / 'grid_consolidated.csv'
grid = pd.read_csv(grid_path) if grid_path.exists() else None
if grid is not None:
    print(f'🔌 Grid substations (NB05): {len(grid):,} (for DSO summary)')

validation_path = DATA_DIR / 'station_validation_metrics.csv'
validation = pd.read_csv(validation_path) if validation_path.exists() else None
if validation is not None:
    print(f'📈 Validation metrics (NB07b): {len(validation):,} stations')

# ── Quick data preview ────────────────────────────────────────────────────────
print('\n── Stations with grid status (first 5 rows) ──')
stations.head()

## Step 2: Assemble File_1 — Global Network KPIs

One row summarising the entire network proposal:
- `total_proposed_stations` — from NB07/NB08 (count of File_2 rows)
- `total_existing_stations_baseline` — from NB04 (interurban fast chargers ≥50 kW)
- `total_friction_points` — from NB08 (count of File_3 rows: Moderate + Congested)
- `total_ev_projected_2027` — **mandatory constant**: 2,498,159

In [ ]:
file1 = pd.DataFrame([{
    'total_proposed_stations': len(stations),
    'total_existing_stations_baseline': total_existing,
    'total_friction_points': len(friction),
    'total_ev_projected_2027': EV_FLEET_2027,
}])

# Enforce column order from schema
file1 = file1[FILE_1_COLUMNS]

# Cast to int (avoid any float artifacts)
for col in FILE_1_COLUMNS:
    file1[col] = file1[col].astype(int)

print('📋 File_1 — Global Network KPIs:')
for col, val in file1.iloc[0].items():
    print(f'   {col}: {val:,}')

file1

## Step 3: Assemble File_2 — Proposed Charging Locations

Schema: `location_id, latitude, longitude, route_segment, n_chargers_proposed, grid_status`

Source: `stations_with_grid_status.csv` (NB08 output). Column mapping handles upstream name variations.

In [ ]:
file2_source = stations.copy()

# ── Defensive column mapping (handle upstream name variations) ────────────────
col_map = {}
if 'route_segment' not in file2_source.columns and 'Carretera' in file2_source.columns:
    col_map['Carretera'] = 'route_segment'
if 'n_chargers_proposed' not in file2_source.columns and 'n_chargers_needed' in file2_source.columns:
    col_map['n_chargers_needed'] = 'n_chargers_proposed'
if col_map:
    file2_source = file2_source.rename(columns=col_map)
    print(f'   Column renames applied: {col_map}')

# ── Validate all required columns exist ──────────────────────────────────────
missing_f2 = [c for c in FILE_2_COLUMNS if c not in file2_source.columns]
if missing_f2:
    raise ValueError(f'File_2 missing columns after mapping: {missing_f2}. '
                     f'Available: {list(file2_source.columns)}')

# ── Select and order columns per schema ──────────────────────────────────────
file2 = file2_source[FILE_2_COLUMNS].copy()

print(f'📋 File_2 — Proposed Stations: {len(file2):,} rows')
print(f'   Total chargers: {file2["n_chargers_proposed"].sum():,}')
print(f'   Total installed capacity: {file2["n_chargers_proposed"].sum() * POWER_PER_CHARGER_KW:,} kW')
print(f'\n   Grid status breakdown:')
print(file2['grid_status'].value_counts().to_string())
print(f'\n   Routes served: {file2["route_segment"].nunique()} unique roads')
print(f'   {", ".join(sorted(file2["route_segment"].unique()))}')

file2

## Step 4: Assemble File_3 — Friction Points

Schema: `bottleneck_id, latitude, longitude, route_segment, distributor_network, estimated_demand_kw, grid_status`

Friction points are stations where grid capacity is insufficient (Moderate or Congested). These highlight locations requiring grid infrastructure investment before charger deployment.

**Note:** NB08 now preserves the nearest distributor label even for stations with no substation inside the 25 km economic search radius. Those sites remain `Congested` because the connection is too remote, but the submission files no longer need any manual DSO overrides.

In [ ]:
file3_source = friction.copy()

# ── Rename location_id → bottleneck_id for File_3 schema ─────────────────────
if 'location_id' in file3_source.columns and 'bottleneck_id' not in file3_source.columns:
    file3_source = file3_source.rename(columns={'location_id': 'bottleneck_id'})

# ── Rename route_segment if needed ───────────────────────────────────────────
if 'route_segment' not in file3_source.columns and 'Carretera' in file3_source.columns:
    file3_source = file3_source.rename(columns={'Carretera': 'route_segment'})

# ── Ensure estimated_demand_kw uses the correct formula ──────────────────────
if 'n_chargers_proposed' in file3_source.columns:
    file3_source['estimated_demand_kw'] = file3_source['n_chargers_proposed'] * POWER_PER_CHARGER_KW

# ── NB08 should already provide valid distributor labels, even for >25 km sites ─
invalid_dso_mask = ~file3_source['distributor_network'].isin(VALID_DISTRIBUTORS)
if invalid_dso_mask.any():
    bad = file3_source.loc[invalid_dso_mask, ['bottleneck_id', 'route_segment', 'distributor_network']]
    raise ValueError(
        'File_3 contains invalid distributor_network values. '
        'Re-run NB08 so unmatched sites inherit the nearest DSO label. '
        f'Problem rows:\n{bad.to_string(index=False)}'
    )

# ── Validate all required columns exist ──────────────────────────────────────
missing_f3 = [c for c in FILE_3_COLUMNS if c not in file3_source.columns]
if missing_f3:
    raise ValueError(f'File_3 missing columns after mapping: {missing_f3}. '
                     f'Available: {list(file3_source.columns)}')

# ── Select and order columns per schema ──────────────────────────────────────
file3 = file3_source[FILE_3_COLUMNS].copy()

print(f'\n📋 File_3 — Friction Points: {len(file3):,} rows')
print(f'   Total estimated demand: {file3["estimated_demand_kw"].sum():,.0f} kW '
      f'({file3["estimated_demand_kw"].sum() / 1000:,.1f} MW)')
print(f'\n   Grid status breakdown:')
print(file3['grid_status'].value_counts().to_string())
print(f'\n   Distributor breakdown:')
print(file3['distributor_network'].value_counts().to_string())

file3


## Step 5: Full Compliance Validation

Every check from the compliance table in the header is enforced here. A single failure aborts the notebook — no partial outputs.

In [ ]:
errors = []
warnings = []

# ── 1. Schema: exact column names and order ──────────────────────────────────
if list(file1.columns) != FILE_1_COLUMNS:
    errors.append(f'File_1 columns mismatch: got {list(file1.columns)}, expected {FILE_1_COLUMNS}')
if list(file2.columns) != FILE_2_COLUMNS:
    errors.append(f'File_2 columns mismatch: got {list(file2.columns)}, expected {FILE_2_COLUMNS}')
if list(file3.columns) != FILE_3_COLUMNS:
    errors.append(f'File_3 columns mismatch: got {list(file3.columns)}, expected {FILE_3_COLUMNS}')

# ── 2. Mandatory EV fleet value ──────────────────────────────────────────────
ev_val = int(file1['total_ev_projected_2027'].iloc[0])
if ev_val != EV_FLEET_2027:
    errors.append(f'total_ev_projected_2027 = {ev_val:,}, MUST be {EV_FLEET_2027:,}')

# ── 3. estimated_demand_kw formula ───────────────────────────────────────────
if len(file3) > 0:
    # Derive from stations data (which has n_chargers_proposed)
    f3_with_chargers = file3.merge(
        stations[['location_id', 'n_chargers_proposed']].rename(
            columns={'location_id': 'bottleneck_id'}),
        on='bottleneck_id', how='left'
    )
    expected_kw = f3_with_chargers['n_chargers_proposed'] * POWER_PER_CHARGER_KW
    mismatch = (file3['estimated_demand_kw'] != expected_kw)
    if mismatch.any():
        bad_ids = file3.loc[mismatch, 'bottleneck_id'].tolist()
        errors.append(f'estimated_demand_kw ≠ n_chargers × {POWER_PER_CHARGER_KW} for: {bad_ids}')

# ── 4. File_3 must not contain Sufficient ────────────────────────────────────
if len(file3) > 0 and 'Sufficient' in file3['grid_status'].values:
    n_suff = (file3['grid_status'] == 'Sufficient').sum()
    errors.append(f'File_3 contains {n_suff} row(s) with grid_status="Sufficient" (prohibited)')

# ── 5. Cross-count consistency ───────────────────────────────────────────────
f1_proposed = int(file1['total_proposed_stations'].iloc[0])
if f1_proposed != len(file2):
    errors.append(f'File_1.total_proposed_stations ({f1_proposed}) ≠ len(File_2) ({len(file2)})')

f1_friction = int(file1['total_friction_points'].iloc[0])
if f1_friction != len(file3):
    errors.append(f'File_1.total_friction_points ({f1_friction}) ≠ len(File_3) ({len(file3)})')

# ── 6. Valid grid_status values (File_2) ─────────────────────────────────────
if len(file2) > 0:
    invalid_f2 = set(file2['grid_status'].unique()) - set(VALID_GRID_STATUSES_FILE2)
    if invalid_f2:
        errors.append(f'File_2 invalid grid_status values: {invalid_f2}')

# ── 7. Valid grid_status values (File_3) ─────────────────────────────────────
if len(file3) > 0:
    invalid_f3 = set(file3['grid_status'].unique()) - set(VALID_GRID_STATUSES_FILE3)
    if invalid_f3:
        errors.append(f'File_3 invalid grid_status values: {invalid_f3}')

# ── 7b. Valid distributor_network values (File_3) ────────────────────────────
if len(file3) > 0:
    invalid_dso = set(file3['distributor_network'].unique()) - set(VALID_DISTRIBUTORS)
    if invalid_dso:
        errors.append(f'File_3 invalid distributor_network values: {invalid_dso}')

# ── 8. Geographic bounds (Spain mainland + Balearic/Canary) ──────────────────
for label, df_check, lat_col, lon_col in [
    ('File_2', file2, 'latitude', 'longitude'),
    ('File_3', file3, 'latitude', 'longitude'),
]:
    if len(df_check) > 0:
        if not df_check[lat_col].between(27.0, 44.0).all():
            errors.append(f'{label} latitude out of Spain range [27, 44]')
        if not df_check[lon_col].between(-19.0, 5.0).all():
            errors.append(f'{label} longitude out of Spain range [-19, 5]')

# ── Additional warnings (non-blocking) ───────────────────────────────────────
if len(file2) > 0 and len(file3) == len(file2):
    warnings.append('All proposed stations are friction points (100% grid-constrained) — '
                     'this is correct given NB08 results but worth highlighting in the report')

if len(file2) > 0 and (file2['grid_status'] == 'Sufficient').sum() == 0:
    warnings.append('No stations have Sufficient grid status — reinforces the grid saturation narrative')

# ── Report ───────────────────────────────────────────────────────────────────
if errors:
    print('❌ VALIDATION FAILED:')
    for e in errors:
        print(f'   ✗ {e}')
    raise AssertionError(f'{len(errors)} compliance check(s) failed — fix before submission')
else:
    print('✅ ALL COMPLIANCE CHECKS PASSED')
    print(f'   ✓ File_1: {len(file1)} row, columns match schema')
    print(f'   ✓ File_2: {len(file2):,} rows, columns match schema')
    print(f'   ✓ File_3: {len(file3):,} rows, no Sufficient, valid DSOs')
    print(f'   ✓ total_ev_projected_2027 = {EV_FLEET_2027:,}')
    print(f'   ✓ estimated_demand_kw = n_chargers × {POWER_PER_CHARGER_KW}')
    print(f'   ✓ Cross-counts consistent (File_1 ↔ File_2/File_3)')
    print(f'   ✓ All coordinates within Spain bounding box')

if warnings:
    print(f'\n📝 Notes ({len(warnings)}):')
    for w in warnings:
        print(f'   • {w}')

## Step 6: Save Output Files

In [ ]:
file1.to_csv(OUTPUT_DIR / 'File_1.csv', index=False)
file2.to_csv(OUTPUT_DIR / 'File_2.csv', index=False)
file3.to_csv(OUTPUT_DIR / 'File_3.csv', index=False)

print('💾 Output files saved:')
for fname, df in [('File_1.csv', file1), ('File_2.csv', file2), ('File_3.csv', file3)]:
    fpath = OUTPUT_DIR / fname
    size_bytes = fpath.stat().st_size
    print(f'   {fname} — {len(df):,} row(s), {size_bytes:,} bytes')

# ── Verify round-trip (read back and compare) ────────────────────────────────
f1_rt = pd.read_csv(OUTPUT_DIR / 'File_1.csv')
f2_rt = pd.read_csv(OUTPUT_DIR / 'File_2.csv')
f3_rt = pd.read_csv(OUTPUT_DIR / 'File_3.csv')

assert list(f1_rt.columns) == FILE_1_COLUMNS, 'File_1 round-trip column mismatch'
assert list(f2_rt.columns) == FILE_2_COLUMNS, 'File_2 round-trip column mismatch'
assert list(f3_rt.columns) == FILE_3_COLUMNS, 'File_3 round-trip column mismatch'
assert len(f2_rt) == len(file2), 'File_2 round-trip row count mismatch'
assert len(f3_rt) == len(file3), 'File_3 round-trip row count mismatch'

print('\n✅ Round-trip verification passed (read-back matches in-memory DataFrames)')

## Step 7: Executive Summary — Network at a Glance

Key figures for the analytical report and pitch deck. This cell computes derived KPIs that go beyond the three submission files to tell the investment story.

In [ ]:
total_chargers = int(file2['n_chargers_proposed'].sum())
total_capacity_kw = total_chargers * POWER_PER_CHARGER_KW
total_capacity_mw = total_capacity_kw / 1000
n_routes = file2['route_segment'].nunique()

# Grid investment breakdown from NB08 enriched data
grid_detail = stations[['location_id', 'route_segment', 'n_chargers_proposed',
                         'available_capacity_mw', 'distributor_network',
                         'connection_distance_km', 'connection_tier',
                         'grid_status', 'estimated_demand_kw']].copy()
grid_detail['estimated_demand_mw'] = grid_detail['estimated_demand_kw'] / 1000
grid_detail['capacity_gap_mw'] = (grid_detail['estimated_demand_mw']
                                   - grid_detail['available_capacity_mw']).clip(lower=0)

# Utilization metrics from NB07b (if available)
util_summary = ''
if validation is not None and 'avg_utilization' in validation.columns:
    avg_util = validation['avg_utilization'].mean()
    peak_util = validation['peak_utilization'].mean() if 'peak_utilization' in validation.columns else None
    queue_dist = validation['queue_risk'].value_counts() if 'queue_risk' in validation.columns else None
    util_summary = f'\n   Avg utilization (baseline): {avg_util:.1%}'
    if peak_util is not None:
        util_summary += f'  |  Peak: {peak_util:.1%}'

# Grid saturation context (from NB05 data)
grid_summary = ''
if grid is not None:
    n_substations = len(grid)
    n_congested = (grid['grid_status'] == 'Congested').sum() if 'grid_status' in grid.columns else 0
    pct_congested = n_congested / n_substations * 100 if n_substations > 0 else 0
    grid_summary = (f'\n   National grid context: {n_congested:,}/{n_substations:,} substations '
                    f'congested ({pct_congested:.0f}%)')

print('=' * 70)
print('  EXECUTIVE SUMMARY — Iberdrola EV Charging Network 2027')
print('=' * 70)
print()
print(f'  📊 NETWORK SCALE')
print(f'   Proposed new stations:         {len(file2):,}')
print(f'   Total new chargers:            {total_chargers:,} × {POWER_PER_CHARGER_KW} kW')
print(f'   Total new capacity:            {total_capacity_mw:.1f} MW')
print(f'   Existing baseline stations:    {total_existing:,} (≥50 kW interurban)')
print(f'   Routes served:                 {n_routes}')
print()
print(f'  🔋 EV FLEET PROJECTION')
print(f'   Projected 2027 fleet:          {EV_FLEET_2027:,} EVs')
print(f'   Chargers per 1,000 EVs:        {total_chargers / EV_FLEET_2027 * 1000:.3f}')
print()
print(f'  ⚡ GRID VIABILITY')
print(f'   Friction points:               {len(file3):,} ({len(file3)/len(file2)*100:.0f}% of proposed)')
print(f'   Total grid capacity gap:       {grid_detail["capacity_gap_mw"].sum():.2f} MW')
print(f'   Stations needing new infra:    {(grid_detail["connection_tier"] == "none").sum()}')
if grid_summary:
    print(grid_summary)
if util_summary:
    print(f'\n  📈 STATION UTILIZATION (NB07b){util_summary}')

print()
print(f'  🗺️  STATION DETAILS')
print(f'   {"ID":<10} {"Route":<10} {"Chargers":>8} {"kW":>7} {"Grid":>10} {"DSO":<10} {"Tier":<10}')
print(f'   {"─"*10} {"─"*10} {"─"*8} {"─"*7} {"─"*10} {"─"*10} {"─"*10}')
for _, r in grid_detail.iterrows():
    tier = r.get('connection_tier', 'N/A')
    dso = r.get('distributor_network', 'N/A')
    print(f'   {r["location_id"]:<10} {r["route_segment"]:<10} {r["n_chargers_proposed"]:>8} '
          f'{r["estimated_demand_kw"]:>7,.0f} {r["grid_status"]:>10} {dso:<10} {tier:<10}')

print()
print('=' * 70)
print('  Files ready for submission: output/File_1.csv, File_2.csv, File_3.csv')
print('  Next: NB10 → visualization/bi_map.html')
print('=' * 70)

## Step 8: DSO Investment Summary

Per-distributor breakdown of grid upgrade requirements — key input for the pitch narrative on Iberdrola's investment opportunity.

In [ ]:
# Per-DSO summary using File_3 (after distributor overrides)
dso_summary = (
    file3
    .merge(
        stations[['location_id', 'n_chargers_proposed', 'available_capacity_mw',
                  'connection_distance_km', 'connection_tier']]
        .rename(columns={'location_id': 'bottleneck_id'}),
        on='bottleneck_id', how='left'
    )
    .groupby('distributor_network')
    .agg(
        n_friction_points=('bottleneck_id', 'count'),
        total_chargers=('n_chargers_proposed', 'sum'),
        total_demand_kw=('estimated_demand_kw', 'sum'),
        avg_capacity_mw=('available_capacity_mw', 'mean'),
        avg_connection_km=('connection_distance_km', 'mean'),
    )
    .sort_values('n_friction_points', ascending=False)
)
dso_summary['total_demand_mw'] = dso_summary['total_demand_kw'] / 1000

print('⚡ Grid Upgrade Requirements by DSO')
print('=' * 65)
for dso, row in dso_summary.iterrows():
    print(f'\n  {dso}:')
    print(f'    Friction points:        {int(row["n_friction_points"])}')
    print(f'    Chargers needed:        {int(row["total_chargers"])}')
    print(f'    Power demand:           {row["total_demand_mw"]:.1f} MW')
    print(f'    Avg available capacity: {row["avg_capacity_mw"]:.2f} MW')
    conn_km = row["avg_connection_km"]
    print(f'    Avg connection dist:    {conn_km:.1f} km' if not pd.isna(conn_km) else
          f'    Avg connection dist:    N/A (no substation in range)')

# Highlight Iberdrola (i-DE) investment opportunity
ide_row = dso_summary.loc['i-DE'] if 'i-DE' in dso_summary.index else None
if ide_row is not None:
    print(f'\n{"─" * 65}')
    print(f'  💡 Iberdrola (i-DE) opportunity: {ide_row["total_demand_mw"]:.1f} MW across '
          f'{int(ide_row["n_friction_points"])} locations')
    print(f'     → Grid reinforcement + charger deployment in i-DE concession territory')

# Save the DSO summary as supplementary output
dso_summary.to_csv(OUTPUT_DIR / 'dso_investment_summary.csv')
print(f'\n💾 Saved: output/dso_investment_summary.csv')